# Explainability in BRB systems

A key advantage of BRB systems over black-box models is that **every intermediate quantity in the inference pipeline has a clear physical or logical meaning**. This notebook demonstrates how to extract and visualize these quantities to explain individual predictions.

For a formal treatment of BRB interpretability, see **Sachan et al. (2020), "Global and local interpretability of belief rule base."** The INFRINGER method by **Misitano (2020)** uses this interpretability for preference modeling in multi-objective optimization, where explaining *why* a model prefers one solution over another is essential for decision maker trust.

```
# requires: pip install matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from desdeo_brb import BRBModel, RuleBase
from desdeo_brb.utils import build_rule_antecedent_indices

## Setup: trained $f(x_1, x_2) = x_1 + x_2$ model

In [ ]:
prv = [np.array([0.0, 1.0, 2.0]), np.array([0.0, 1.0, 2.0])]
crv = np.array([0.0, 1.0, 2.0, 3.0, 4.0])

model = BRBModel(prv, crv, initial_rule_fn=lambda x: x[0] + x[1])
print(f"{model.rule_base.n_rules} rules, {model.rule_base.n_attributes} attributes")

## Explaining a prediction

Let's predict at the point $(1.3, 0.7)$ where $f = 2.0$, and inspect every layer of the inference.

In [ ]:
X_point = np.array([[1.3, 0.7]])
result = model.predict(X_point)

print(f"Input:     x1={X_point[0, 0]}, x2={X_point[0, 1]}")
print(f"True:      f(x) = {X_point[0, 0] + X_point[0, 1]:.1f}")
print(f"Predicted: {result.output[0]:.4f}")

### Which rules fired?

The activation weights tell us how strongly each rule matches the input. Since the input is between referential values, multiple rules are activated.

In [ ]:
weights = result.activation_weights[0]
indices = model.rule_base.rule_antecedent_indices

# Build rule labels
rule_labels = []
for k in range(model.rule_base.n_rules):
    x1 = prv[0][indices[k, 0]]
    x2 = prv[1][indices[k, 1]]
    rule_labels.append(f"R{k + 1}\n({x1:.0f},{x2:.0f})")

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#d62728" if w > 0.05 else "#aec7e8" for w in weights]
ax.bar(range(len(weights)), weights, color=colors)
ax.set_xticks(range(len(weights)))
ax.set_xticklabels(rule_labels, fontsize=8)
ax.set_ylabel("Activation weight")
ax.set_title(f"Rule activation weights for input ({X_point[0, 0]}, {X_point[0, 1]})")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Print the active rules
print("Active rules (weight > 1%):")
for k in np.where(weights > 0.01)[0]:
    x1 = prv[0][indices[k, 0]]
    x2 = prv[1][indices[k, 1]]
    print(f"  Rule {k + 1}: x1={x1:.0f}, x2={x2:.0f}, weight={weights[k]:.4f}")

### What does the output belief distribution look like?

The combined belief degrees show how the model distributes belief across the consequent referential values.

In [ ]:
beliefs = result.combined_belief_degrees[0]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(crv, beliefs, width=0.6, color="steelblue", edgecolor="black")
ax.axvline(
    result.output[0],
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Output = {result.output[0]:.2f}",
)
ax.set_xlabel("Consequent referential values")
ax.set_ylabel("Belief degree")
ax.set_title("Combined belief distribution")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### Mapping dominant rules to their meaning

The `dominant_rules()` method returns the indices of the most activated rules. We can map these back to their antecedent values to understand *why* the model made its prediction.

In [ ]:
top_k = 3
top_indices = result.dominant_rules(top_k=top_k)[0]

print(f"Top-{top_k} dominant rules for input ({X_point[0, 0]}, {X_point[0, 1]}):")
print()
for rank, k in enumerate(top_indices, 1):
    x1 = prv[0][indices[k, 0]]
    x2 = prv[1][indices[k, 1]]
    w = weights[k]
    bd = model.rule_base.belief_degrees[k]
    print(f"  #{rank}: Rule {k + 1} (x1={x1:.0f}, x2={x2:.0f})")
    print(f"       Activation weight: {w:.4f}")
    print(f"       Belief degrees: {dict(zip(crv, np.round(bd, 3)))}")
    print()

## Explaining a pipeline leak detection prediction

Using the expert-defined pipeline model from notebook 03 (Chen et al. 2011, Table D.1), we can explain *why* the system detected a potential leak.

In [ ]:
# Reconstruct the pipeline model (same setup as notebook 03)
flow_diff_rv = np.array([-10.0, -4.1, -2.8, -1.79, -0.79, 0.25, 2.0, 3.0])
pressure_diff_rv = np.array([-0.01, -0.008, -0.005, 0.003, 0.0058, 0.008, 0.01])
leak_size_rv = np.array([0.0, 2.0, 4.0, 6.0, 8.0])
prv_pipe = [flow_diff_rv, pressure_diff_rv]

expert_beliefs = np.array(
    [
        [0.00, 0.00, 0.00, 0.00, 1.00],
        [0.00, 0.00, 0.00, 0.30, 0.70],
        [0.00, 0.00, 0.20, 0.80, 0.00],
        [0.00, 0.00, 0.80, 0.20, 0.00],
        [0.65, 0.35, 0.00, 0.00, 0.00],
        [0.85, 0.15, 0.00, 0.00, 0.00],
        [0.95, 0.05, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.10, 0.90, 0.00],
        [0.00, 0.00, 0.70, 0.30, 0.00],
        [0.00, 0.70, 0.30, 0.00, 0.00],
        [0.00, 0.90, 0.10, 0.00, 0.00],
        [0.80, 0.20, 0.00, 0.00, 0.00],
        [0.90, 0.10, 0.00, 0.00, 0.00],
        [0.99, 0.01, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.40, 0.60, 0.00],
        [0.00, 0.00, 0.80, 0.20, 0.00],
        [0.00, 0.30, 0.60, 0.10, 0.00],
        [0.10, 0.80, 0.10, 0.00, 0.00],
        [0.90, 0.10, 0.00, 0.00, 0.00],
        [0.95, 0.05, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.50, 0.50, 0.00],
        [0.00, 0.10, 0.80, 0.10, 0.00],
        [0.00, 0.80, 0.20, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [0.95, 0.05, 0.00, 0.00, 0.00],
        [0.98, 0.02, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.60, 0.40, 0.00],
        [0.00, 0.20, 0.60, 0.20, 0.00],
        [0.10, 0.60, 0.30, 0.00, 0.00],
        [0.90, 0.10, 0.00, 0.00, 0.00],
        [0.95, 0.05, 0.00, 0.00, 0.00],
        [0.99, 0.01, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.60, 0.40, 0.00],
        [0.00, 0.20, 0.70, 0.10, 0.00],
        [0.00, 0.70, 0.30, 0.00, 0.00],
        [0.95, 0.05, 0.00, 0.00, 0.00],
        [0.99, 0.01, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.10, 0.70, 0.20, 0.00],
        [0.00, 0.30, 0.60, 0.10, 0.00],
        [0.10, 0.70, 0.20, 0.00, 0.00],
        [0.98, 0.02, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.10, 0.80, 0.10, 0.00],
        [0.00, 0.30, 0.70, 0.00, 0.00],
        [0.10, 0.80, 0.10, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00, 0.00, 0.00],
    ]
)

n_rules_pipe = 56
rb_pipe = RuleBase(
    precedent_referential_values=prv_pipe,
    consequent_referential_values=leak_size_rv,
    belief_degrees=expert_beliefs,
    rule_weights=np.full(n_rules_pipe, 1.0 / n_rules_pipe),
    attribute_weights=np.ones((n_rules_pipe, 2)),
    rule_antecedent_indices=build_rule_antecedent_indices(prv_pipe),
)
pipe_model = BRBModel(prv_pipe, leak_size_rv, rule_base=rb_pipe)

In [ ]:
# Predict at a leak scenario
X_leak = np.array([[-5.0, -0.007]])
result_leak = pipe_model.predict(X_leak)

flow_labels = ["NL", "NM", "NS", "Z", "PS", "PM", "PL", "PVL"]
press_labels = ["NL", "NM", "NS", "Z", "PS", "PM", "PL"]
leak_labels = ["Zero", "Small", "Medium", "High", "VeryHigh"]

print(f"Scenario: FlowDiff={X_leak[0, 0]}, PressureDiff={X_leak[0, 1]}")
print(f"Predicted LeakSize: {result_leak.output[0]:.2f}")
print()

# Show top-5 activated rules
pipe_indices = rb_pipe.rule_antecedent_indices
pipe_weights = result_leak.activation_weights[0]
top5 = result_leak.dominant_rules(top_k=5)[0]

print("Top-5 activated rules:")
for rank, k in enumerate(top5, 1):
    fi = pipe_indices[k, 0]
    pi = pipe_indices[k, 1]
    w = pipe_weights[k]
    bd = rb_pipe.belief_degrees[k]
    dominant_leak = leak_labels[np.argmax(bd)]
    print(
        f"  #{rank}: Rule {k + 1} (FlowDiff={flow_labels[fi]}, PressureDiff={press_labels[pi]})"
    )
    print(f"       Weight: {w:.4f}, Dominant: {dominant_leak}")
    print(f"       Beliefs: {dict(zip(leak_labels, np.round(bd, 2)))}")

In [ ]:
# Visualize the belief distribution
beliefs_leak = result_leak.combined_belief_degrees[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Activation weights (top 10 rules)
top10 = np.argsort(-pipe_weights)[:10]
top_w = pipe_weights[top10]
top_labels = [
    f"R{k + 1}\n{flow_labels[pipe_indices[k, 0]]},{press_labels[pipe_indices[k, 1]]}"
    for k in top10
]
axes[0].barh(range(len(top10)), top_w[::-1], color="steelblue")
axes[0].set_yticks(range(len(top10)))
axes[0].set_yticklabels(top_labels[::-1], fontsize=8)
axes[0].set_xlabel("Activation weight")
axes[0].set_title("Top-10 activated rules")
axes[0].grid(True, alpha=0.3, axis="x")

# Belief distribution
axes[1].bar(leak_labels, beliefs_leak, color="coral", edgecolor="black")
axes[1].axvline(
    leak_labels[np.argmax(beliefs_leak)], color="red", alpha=0.3, linewidth=10
)
axes[1].set_ylabel("Belief degree")
axes[1].set_title(
    f"Output belief distribution (predicted LeakSize={result_leak.output[0]:.1f})"
)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Explanations for downstream applications

The ability to explain individual predictions makes BRB systems valuable in contexts where trust and transparency matter:

- **Fault diagnosis**: "The system detected a potential leak because rules matching high negative flow difference were strongly activated."
- **Risk assessment**: "The elevated risk score is driven by rules 3 and 7, which correspond to high temperature and low pressure conditions."
- **Preference modeling**: The INFRINGER method (Misitano 2020) uses BRB systems to learn a decision maker's preferences in multi-objective optimization. Because the learned preference model is a BRB, the optimizer can explain *why* it ranks one solution above another by showing which preference rules were activated.

In all cases, the explainability is intrinsic to the model structure — it does not require post-hoc explanation techniques like LIME or SHAP.